## 1. Obtención de los datos

Descarga del conjunto de datos meteorológicos de la estación de A Coruña / Alvedro correspondiente al período 2021-2025.

Los datos proceden de Meteostat y han sido modificados con fines didácticos para incorporar problemas de calidad.

In [5]:
import requests
from pathlib import Path

URL = "https://raw.githubusercontent.com/bigdatawirtz/datasets/main/datasets/meteo_alvedro_2021_2025.csv"

destino = Path("../datos/raw/meteo_alvedro_2021_2025.csv")

# Crear las carpetas si todavía no existen
destino.parent.mkdir(parents=True, exist_ok=True)

# Descargar el archivo
respuesta = requests.get(URL, timeout=30)
respuesta.raise_for_status()

# Guardar los bytes originales sin modificarlos
destino.write_bytes(respuesta.content)

print("Archivo descargado correctamente.")
print("Tamaño:", destino.stat().st_size, "bytes")

Archivo descargado correctamente.
Tamaño: 81825 bytes


2.1. Localización e inspección del archivo original

In [2]:
from pathlib import Path

ruta = Path("../datos/raw/meteo_alvedro_2021_2025.csv")

print("¿Existe el archivo?", ruta.exists())

¿Existe el archivo? True


2.2. Inspección de bytes y codificación

In [3]:
with open(ruta, "rb") as archivo:
    primeros_bytes = archivo.read(200)

print(primeros_bytes)

b'Servizo de exportaci\xf3n de datos meteorol\xf3xicos\nEstaci\xf3n: A Coru\xf1a / Alvedro (aeroporto) \xb7 Indicativo: 08002\nLatitude: 43,3000 \xb7 Lonxitude: -8,3833 \xb7 Altitude: 97 m\nPer\xedodo: 01/01/2021 - 31/12/2025\nFon'


2.3. Exploración de las primeras y últimas líneas

In [4]:
with open(ruta, "r", encoding="latin-1") as archivo:
    lineas = archivo.readlines()

print("Número total de líneas:", len(lineas))

print("\nPRIMERAS 10 LÍNEAS:")
for linea in lineas[:10]:
    print(repr(linea))

print("\nÚLTIMAS 5 LÍNEAS:")
for linea in lineas[-5:]:
    print(repr(linea))

Número total de líneas: 1819

PRIMERAS 10 LÍNEAS:
'Servizo de exportación de datos meteorolóxicos\n'
'Estación: A Coruña / Alvedro (aeroporto) · Indicativo: 08002\n'
'Latitude: 43,3000 · Lonxitude: -8,3833 · Altitude: 97 m\n'
'Período: 01/01/2021 - 31/12/2025\n'
'Fonte orixinal: Meteostat (meteostat.net). Ficheiro modificado con fins didácticos.\n'
'\n'
'Data;Temp. media (ºC);Temp. mínima (ºC);Temp. máxima (ºC);Humidade relativa (%);Precipitación (mm);Vento medio (km/h);Presión (hPa)\n'
'2021-01-01;5,9;3,3;9,1;87;12,2;11,0;1.015,7\n'
'2021-01-02;5,5;3,1;9,7;91;9,8;6,4;1.019,8\n'
'2021-01-03;5,6;3,0;8,4;92;12,4;8,5;1.018,9\n'

ÚLTIMAS 5 LÍNEAS:
'2025-06-10;20,8;15,6;27,6;80;Ip;6,2;1.015,7\n'
'2021-04-16;10,0;3,6;17,2;56;0,0;11,7;1.020,1\n'
'2022-05-21;19,9;10,9;24,7;81;2,9;5,8;1.015,6\n'
'2025-10-12;17,6;9,9;24,9;74;0,0;5,8;1.018,8\n'
'17/08/2023;20,1;14,2;26,7;77;0,5;9,3;1.014,7\n'


### Observaciones iniciales

El archivo contiene 1819 líneas, incluyendo metadatos iniciales sobre la estación meteorológica.

La lectura preliminar con codificación Latin-1 permite visualizar correctamente los caracteres especiales.

Los campos parecen estar separados por punto y coma (`;`) y los valores numéricos utilizan coma decimal (`,`).

Las últimas líneas no están ordenadas cronológicamente, por lo que será necesario revisar las fechas durante la limpieza.

In [6]:
# Separar los registros de los metadatos
registros = lineas[7:]

# Mostrar las columnas del CSV
columnas = lineas[6].strip().split(";")

print("Columnas del archivo:")
for columna in columnas:
    print("-", columna)

# Buscar posibles valores ausentes
print("\nEjemplos de registros con valores sospechosos:")

for registro in registros:
    if "NA" in registro or "Ip" in registro or ";;" in registro:
        print(registro.strip())
        break

Columnas del archivo:
- Data
- Temp. media (ºC)
- Temp. mínima (ºC)
- Temp. máxima (ºC)
- Humidade relativa (%)
- Precipitación (mm)
- Vento medio (km/h)
- Presión (hPa)

Ejemplos de registros con valores sospechosos:
2021-01-15;8,2;1,9;12,7;88;Ip;5,6;1.031,4


### Observaciones sobre el formato

El archivo contiene 8 columnas con información meteorológica diaria.

Se ha identificado que:
- Las columnas están separadas por punto y coma (`;`).
- Los valores numéricos utilizan coma decimal y, en algunos casos, punto como separador de miles.
- Las primeras 6 líneas contienen metadatos, y la séptima corresponde a las cabeceras.
- Aparece el código `Ip` en la precipitación, cuyo significado y tratamiento deben verificarse.
- Las últimas filas no están ordenadas cronológicamente.

In [7]:
import pandas as pd

df = pd.read_csv(
    ruta,
    encoding="latin-1",
    sep=";",
    decimal=",",
    thousands=".",
    skiprows=6
)

df.head()

,Data,Temp. media (ºC),Temp. mínima (ºC),Temp. máxima (ºC),Humidade relativa (%),Precipitación (mm),Vento medio (km/h),Presión (hPa)
0,2021-01-01,5.9,3.3,9.1,87,"12,2",11.0,1015.7
1,2021-01-02,5.5,3.1,9.7,91,"9,8",6.4,1019.8
2,2021-01-03,5.6,3.0,8.4,92,"12,4",8.5,1018.9
3,2021-01-04,3.6,-0.3,6.8,92,"4,7",7.8,1015.4
4,2021-01-05,2.8,-1.5,9.4,94,"1,2",6.2,1014.6


### 2.4. Comprobación de tipos de datos

Se comprueban los tipos asignados por pandas a cada columna y la cantidad de valores no nulos, con el objetivo de identificar posibles problemas de interpretación.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1812 entries, 0 to 1811
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Data                   1812 non-null   str    
 1   Temp. media (ºC)       1803 non-null   float64
 2   Temp. mínima (ºC)      1806 non-null   float64
 3   Temp. máxima (ºC)      1803 non-null   float64
 4   Humidade relativa (%)  1812 non-null   int64  
 5   Precipitación (mm)     1804 non-null   str    
 6   Vento medio (km/h)     1812 non-null   float64
 7   Presión (hPa)          1811 non-null   float64
dtypes: float64(5), int64(1), str(2)
memory usage: 113.4 KB


### Observaciones

El DataFrame contiene 1812 registros y 8 columnas.

Se detectan dos columnas con tipos que requieren revisión:
- `Data`: se ha interpretado como texto en lugar de fecha.
- `Precipitación (mm)`: se ha interpretado como texto debido a la presencia de valores no numéricos.

También se observan valores ausentes en varias columnas de temperatura, precipitación y presión.

Será necesario investigar estos problemas antes de realizar el análisis.

### 2.5. Análisis de valores especiales en precipitación

Se analizan los valores de la columna de precipitación para identificar códigos no numéricos que impiden su correcta interpretación.

Antes de realizar cualquier conversión, se comprueba qué valores aparecen y cuántas veces se repiten.

In [9]:
# Seleccionar la columna de precipitación
precipitacion = df["Precipitación (mm)"]

# Mostrar los valores más frecuentes
print("Valores más frecuentes:")
print(precipitacion.value_counts().head(20))

# Comprobar valores ausentes
print("\nValores nulos:", precipitacion.isna().sum())

Valores más frecuentes:
Precipitación (mm)
0,0    809
Ip     121
0,2     49
0,3     34
0,4     29
0,5     28
0,7     23
0,9     21
0,8     19
1,2     16
1,0     15
2,1     14
0,6     14
1,9     14
1,6     13
1,1     12
3,6     12
1,4     12
2,9     12
2,6     12
Name: count, dtype: int64

Valores nulos: 8


### 2.6. Identificación de valores no numéricos

Se identifican los valores de precipitación que no pueden convertirse a números, con el objetivo de localizar posibles códigos especiales o errores de formato.

In [10]:
# Sustituir la coma decimal por un punto
precip_texto = precipitacion.str.replace(",", ".", regex=False)

# Intentar convertir los valores a números
precip_numerica = pd.to_numeric(precip_texto, errors="coerce")

# Detectar valores que no se han podido convertir
no_numericos = precipitacion[
    precip_numerica.isna() & precipitacion.notna()
]

print("Valores no numéricos encontrados:")
print(no_numericos.value_counts())

Valores no numéricos encontrados:
Precipitación (mm)
Ip    121
Name: count, dtype: int64


### 2.7. Análisis de valores ausentes

Se identifican los registros que contienen valores nulos y las variables afectadas.

El objetivo es determinar si los valores ausentes corresponden a errores puntuales o a registros incompletos antes de decidir cómo tratarlos.

In [11]:
# Contar los valores nulos de cada columna
print("Valores nulos por columna:")
print(df.isna().sum())

# Mostrar las filas que contienen al menos un valor nulo
filas_nulas = df[df.isna().any(axis=1)]

print("\nRegistros con valores ausentes:")
display(filas_nulas)

Valores nulos por columna:
Data                     0
Temp. media (ºC)         9
Temp. mínima (ºC)        6
Temp. máxima (ºC)        9
Humidade relativa (%)    0
Precipitación (mm)       8
Vento medio (km/h)       0
Presión (hPa)            1
dtype: int64

Registros con valores ausentes:


,Data,Temp. media (ºC),Temp. mínima (ºC),Temp. máxima (ºC),Humidade relativa (%),Precipitación (mm),Vento medio (km/h),Presión (hPa)
164,2021-06-16,18.0,15.7,19.5,90,NaN,7.4,1011.6
175,2021-06-27,NaN,10.2,18.4,79,Ip,13.8,1017.8
211,2021-08-02,NaN,12.8,21.2,79,"0,0",11.2,1018.8
295,2021-10-25,14.1,NaN,18.5,87,Ip,8.7,1023.3
313,2021-11-12,10.3,3.3,16.4,90,NaN,7.5,1021.5
338,2021-12-08,9.0,4.5,NaN,79,"5,9",22.0,1016.7
350,2021-12-20,13.6,NaN,16.5,62,"1,1",15.8,1015.9
376,2022-01-15,6.7,NaN,14.0,72,"0,0",12.5,1026.9
408,2022-02-16,14.2,11.8,NaN,80,"0,0",13.9,1027.4
506,2022-05-25,NaN,8.7,18.4,81,"0,8",10.1,1025.4
